# Phase 5 Gradio Read-Only User Experience

This notebook launches a local Gradio interface for the validated Phase 5 read-only flow. The user signs in with Microsoft Entra to obtain Token A, then requests the synthetic account balance through the unified Solo gateway. Token B and Token C remain agent-owned.

Prerequisite: keep the `solo-unified-live` stack running. This notebook never starts or stops Docker. The Gradio interface binds only to loopback and creates no public share link.

In [ ]:
import html
import json
import os
import re
import uuid
import urllib.error
import urllib.request
from pathlib import Path
from urllib.parse import urlparse

import gradio as gr
import msal
from dotenv import load_dotenv

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "agents").exists() and (REPO_ROOT.parent / "agents").exists():
    REPO_ROOT = REPO_ROOT.parent
load_dotenv(REPO_ROOT / ".env", override=False)


class DemoError(RuntimeError):
    pass


def _base_url() -> str:
    value = os.getenv("SOLO_API_BASE_URL", "http://127.0.0.1:18080").strip().rstrip("/")
    parsed = urlparse(value)
    if parsed.scheme not in {"http", "https"} or parsed.hostname not in {"127.0.0.1", "localhost", "::1"}:
        raise DemoError("The Solo API endpoint must use a loopback address.")
    return value


def _entra_settings() -> tuple[str, str, str]:
    tenant_id = os.getenv("ENTRA_TENANT_ID", "").strip()
    client_id = os.getenv("ENTRA_NOTEBOOK_CLIENT_ID", "").strip()
    scope = os.getenv("ENTRA_AGENT_A_SCOPE", "").strip()
    if not tenant_id or not client_id or not scope:
        raise DemoError("The notebook Entra configuration is incomplete.")
    return tenant_id, client_id, scope


def _device_sign_in_html(flow: object) -> str:
    if not isinstance(flow, dict):
        raise DemoError("Microsoft Entra returned an invalid device sign-in response.")
    verification_uri = flow.get("verification_uri") or flow.get("verification_url")
    user_code = flow.get("user_code")
    if not isinstance(verification_uri, str) or not isinstance(user_code, str):
        raise DemoError("Microsoft Entra did not return a device sign-in link and code.")
    parsed = urlparse(verification_uri)
    allowed_hosts = {"microsoft.com", "www.microsoft.com", "login.microsoft.com", "login.microsoftonline.com"}
    if parsed.scheme != "https" or parsed.hostname not in allowed_hosts:
        raise DemoError("Microsoft Entra returned an unapproved device sign-in link.")
    safe_uri = html.escape(verification_uri, quote=True)
    safe_code = html.escape(user_code)
    return (
        "<div style='padding:12px;border:1px solid #ccc;border-radius:8px'>"
        "<strong>Microsoft Entra sign-in required</strong><br><br>"
        f"<a href='{safe_uri}' target='_blank' rel='noopener noreferrer' "
        "style='display:inline-block;padding:10px 16px;background:#2563eb;color:white;text-decoration:none;border-radius:6px'>"
        "Open Microsoft sign-in</a>"
        f"<p>Enter this one-time code: <code style='font-size:1.2em'><strong>{safe_code}</strong></code></p>"
        "<p>After authentication, this page will update automatically.</p></div>"
    )


def _http_json(url: str, *, method: str, token_a: str, payload: object, flow_id: str | None = None):
    headers = {"Authorization": f"Bearer {token_a}", "Content-Type": "application/json"}
    if flow_id:
        headers["X-Flow-ID"] = flow_id
    request = urllib.request.Request(
        url,
        method=method,
        headers=headers,
        data=json.dumps(payload).encode("utf-8"),
    )
    try:
        with urllib.request.urlopen(request, timeout=90) as response:
            body = response.read().decode("utf-8")
            return response.status, json.loads(body) if body else None, dict(response.headers)
    except urllib.error.HTTPError as exc:
        raise DemoError(f"The protected API returned HTTP {exc.code}.") from exc
    except urllib.error.URLError as exc:
        raise DemoError("The local Solo API is unavailable.") from exc
    except json.JSONDecodeError as exc:
        raise DemoError("The protected API returned an invalid response.") from exc


def _iter_parts(events: object):
    if not isinstance(events, list):
        raise DemoError("The agent response did not match the proven contract.")
    for event in events:
        if not isinstance(event, dict):
            raise DemoError("The agent response did not match the proven contract.")
        content = event.get("content")
        parts = content.get("parts") if isinstance(content, dict) else event.get("parts")
        if parts is None:
            continue
        if not isinstance(parts, list):
            raise DemoError("The agent response did not match the proven contract.")
        yield event, parts


def _tool_payload(part: object) -> dict | None:
    if not isinstance(part, dict):
        return None
    for key in ("functionCall", "functionResponse", "function_call", "function_response", "toolCall", "toolResponse", "tool_call", "tool_response"):
        value = part.get(key)
        if isinstance(value, dict):
            return value
    return None


def _tool_name(payload: dict | None) -> str:
    if not isinstance(payload, dict):
        return ""
    for key in ("name", "tool_name", "toolName"):
        value = payload.get(key)
        if isinstance(value, str):
            return value
    return ""


def _sanitize_result(text: str) -> str:
    value = re.sub(r"\b(?:acct|user)-\d+\b", "[redacted]", text.strip())
    return re.sub(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", "[redacted]", value)


def _response_text(value: object) -> str:
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        for item in value:
            text = _response_text(item)
            if text:
                return text
        return ""
    if isinstance(value, dict):
        for key in ("result", "output", "text", "content"):
            if key in value:
                text = _response_text(value[key])
                if text:
                    return text
    return ""


def _extract_business_result(events: object) -> str:
    event_parts = list(_iter_parts(events))
    tool_calls = []
    tool_responses = []
    for _, parts in event_parts:
        for part in parts:
            payload = _tool_payload(part)
            name = _tool_name(payload).lower()
            if not name:
                continue
            if any(key in part for key in ("functionCall", "function_call", "toolCall", "tool_call")):
                tool_calls.append(name)
            if any(key in part for key in ("functionResponse", "function_response", "toolResponse", "tool_response")):
                tool_responses.append(name)
    if set(tool_calls + tool_responses) - {"delegate_to_agent_b", "get_account_summary"}:
        raise DemoError("An unexpected tool appeared in the response.")
    delegated_call = "delegate_to_agent_b" in tool_calls
    delegated_response = "delegate_to_agent_b" in tool_responses
    if delegated_call != delegated_response:
        raise DemoError("Agent A delegation did not complete consistently.")
    for event, parts in reversed(event_parts):
        if str(event.get("author") or "").lower() != "agent_a":
            continue
        texts = []
        for part in parts:
            if not isinstance(part, dict) or bool(part.get("thought")) or _tool_payload(part) is not None:
                continue
            text = part.get("text")
            if isinstance(text, str) and text.strip():
                texts.append(text.strip())
        if texts:
            return _sanitize_result("\n".join(texts))
    if delegated_response:
        for _, parts in reversed(event_parts):
            for part in reversed(parts):
                if not isinstance(part, dict):
                    continue
                for key in ("functionResponse", "function_response", "toolResponse", "tool_response"):
                    payload = part.get(key)
                    if isinstance(payload, dict) and _tool_name(payload) == "delegate_to_agent_b":
                        text = _response_text(payload.get("response"))
                        if text:
                            return _sanitize_result(text)
    raise DemoError("No final Agent A response was available.")

In [ ]:
def sign_in_with_entra():
    try:
        tenant_id, client_id, scope = _entra_settings()
        client = msal.PublicClientApplication(
            client_id=client_id,
            authority=f"https://login.microsoftonline.com/{tenant_id}",
        )
        flow = client.initiate_device_flow(scopes=[scope])
        yield None, _device_sign_in_html(flow)
        result = client.acquire_token_by_device_flow(flow)
        token_a = result.get("access_token") if isinstance(result, dict) else None
        if not isinstance(token_a, str) or not token_a.strip():
            raise DemoError("Microsoft Entra sign-in did not complete successfully.")
        yield token_a.strip(), "<div style='color:#166534'><strong>Signed in successfully.</strong> Token A is held only in this local Gradio session.</div>"
    except DemoError as exc:
        yield None, f"<div style='color:#991b1b'>Sign-in failed: {html.escape(str(exc))}</div>"
    except Exception:
        yield None, "<div style='color:#991b1b'>Sign-in failed unexpectedly. Check the notebook terminal for diagnostics.</div>"


def _normalize_chat_history(history: object) -> list[dict[str, str]]:
    if not isinstance(history, list):
        return []
    normalized = []
    for item in history:
        role = item.get("role") if isinstance(item, dict) else getattr(item, "role", None)
        content = item.get("content") if isinstance(item, dict) else getattr(item, "content", None)
        if role in {"user", "assistant"} and isinstance(content, str):
            normalized.append({"role": role, "content": content})
    return normalized


def _replace_assistant_message(history: list[dict[str, str]], message: str) -> list[dict[str, str]]:
    updated = [dict(item) for item in history]
    if updated and updated[-1].get("role") == "assistant":
        updated[-1] = {"role": "assistant", "content": message}
    else:
        updated.append({"role": "assistant", "content": message})
    return updated


def _render_activity(completed: list[str], *, current: str | None = None, error: str | None = None, flow_id: str | None = None) -> str:
    lines = [f"✅ {step}" for step in completed]
    if current:
        lines.append(f"⏳ {current}")
    if error:
        lines.append(f"❌ {error}")
    if flow_id:
        lines.append(f"Flow ID: {flow_id}")
    return "\n".join(lines)


def run_agent_prompt(prompt: str, history: object, token_a: str | None, session_id: str | None):
    chat = _normalize_chat_history(history)
    prompt = prompt.strip() if isinstance(prompt, str) else ""
    if not prompt:
        yield chat, "Enter a question for Agent A.", prompt, session_id
        return
    if len(prompt) > 500:
        yield chat, "The prompt is too long for this POC.", prompt, session_id
        return
    if not isinstance(token_a, str) or not token_a.strip():
        yield chat, "Microsoft Entra sign-in has not completed yet.", prompt, session_id
        return
    chat = chat + [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": "Working on your request..."},
    ]
    completed = ["1/5 Prompt validated."]
    yield chat, _render_activity(completed, current="2/5 Preparing the protected Agent A conversation session..."), "", session_id
    try:
        base_url = _base_url()
        flow_id = f"gradio-{uuid.uuid4().hex[:12]}"
        user_id = "user-12345"
        active_session_id = session_id.strip() if isinstance(session_id, str) else ""
        if not active_session_id:
            session_status, session_body, _ = _http_json(
                f"{base_url}/apps/agent_a/users/{user_id}/sessions",
                method="POST",
                token_a=token_a,
                payload={},
            )
            if session_status != 200 or not isinstance(session_body, dict):
                raise DemoError("The Agent A session could not be created.")
            active_session_id = session_body.get("id")
            if not isinstance(active_session_id, str) or not active_session_id.strip():
                raise DemoError("The Agent A session identifier was absent.")
            completed.append("2/5 Agent A conversation session created.")
        else:
            completed.append("2/5 Existing Agent A conversation session reused.")
        yield chat, _render_activity(completed, current="3/5 Agent A is processing the request; banking reads may delegate to Agent B..."), "", active_session_id
        run_status, events, headers = _http_json(
            f"{base_url}/run",
            method="POST",
            token_a=token_a,
            flow_id=flow_id,
            payload={
                "app_name": "agent_a",
                "user_id": user_id,
                "session_id": active_session_id,
                "new_message": {"role": "user", "parts": [{"text": prompt}]},
            },
        )
        if run_status != 200:
            raise DemoError("The Agent A request did not succeed.")
        response_flow_id = next((value for key, value in headers.items() if key.lower() == "x-flow-id"), None)
        if response_flow_id != flow_id:
            raise DemoError("The API did not preserve the request flow identifier.")
        completed.append("3/5 Agent A returned a response.")
        yield chat, _render_activity(completed, current="4/5 Validating response structure and tool boundaries..."), "", active_session_id
        final_result = _extract_business_result(events)
        if "balance" in prompt.lower() and "acct-1001" in prompt.lower() and "12345.67" not in final_result.replace(",", ""):
            raise DemoError("The expected read-only balance signal was absent.")
        completed.append("4/5 Response structure and tool boundaries validated.")
        yield chat, _render_activity(completed, current="5/5 Finalizing the chat response..."), "", active_session_id
        completed.append("5/5 Request completed successfully.")
        yield _replace_assistant_message(chat, final_result), _render_activity(completed, flow_id=flow_id), "", active_session_id
    except DemoError as exc:
        yield _replace_assistant_message(chat, f"Request failed: {exc}"), _render_activity(completed, error=f"Request failed: {exc}"), prompt, session_id
    except Exception:
        message = "Request failed unexpectedly. Check the service logs for diagnostics."
        yield _replace_assistant_message(chat, message), _render_activity(completed, error=message), prompt, session_id


In [ ]:
with gr.Blocks(title="Enterprise Agent POC") as demo:
    gr.Markdown(
        "# Enterprise Agent POC\n"
        "Microsoft Entra sign-in starts automatically. After sign-in completes, enter any question for Agent A and press Enter. Banking reads may delegate to Agent B; writes are refused by the Phase 5 agent policy."
    )
    token_a_state = gr.State(value=None)
    agent_session_state = gr.State(value=None)
    sign_in_status = gr.HTML()
    chat_history = gr.Chatbot(
        label="Conversation with Agent A",
        layout="bubble",
        height=420,
        placeholder="Complete Microsoft Entra sign-in, then ask Agent A a question.",
        buttons=["copy"],
    )
    prompt_input = gr.Textbox(
        label="Your question for Agent A",
        placeholder=(
            "Enter any question for Agent A.\n\n"
            "Read example: What is the balance for synthetic account acct-1001?\n"
            "Transfer example: Transfer 25.00 from synthetic account acct-1001 to acct-2002."
        ),
        lines=4,
        max_lines=8,
        info="Enter adds a new line. Select Send to submit the message.",
        submit_btn="Send",
        interactive=True,
    )
    request_status = gr.Textbox(label="Live request activity", interactive=False, lines=8)
    gr.Markdown("General questions are handled by Agent A. Read-only banking questions may delegate to Agent B. Transfer and write requests reach Agent A but must be refused by the Phase 5 policy; no write tool is permitted.")

    demo.load(
        fn=sign_in_with_entra,
        inputs=None,
        outputs=[token_a_state, sign_in_status],
    )
    prompt_input.submit(
        fn=run_agent_prompt,
        inputs=[prompt_input, chat_history, token_a_state, agent_session_state],
        outputs=[chat_history, request_status, prompt_input, agent_session_state],
    )

demo.queue(default_concurrency_limit=1)
demo.launch(
    inline=False,
    inbrowser=True,
    share=False,
    server_name="127.0.0.1",
    prevent_thread_lock=True,
    show_error=False,
)

## Stop the interface

When finished, run `demo.close()` in a new cell or restart the notebook kernel. Stop the live Docker stack separately with the guarded live runner's `--stop` option.